# GPT-2 Inference Benchmark
Measures tokens/sec, time to first token, and peak GPU memory for GPT-2 small on a T4,
comparing fp32 vs fp16 and KV cache on vs off at batch sizes 1, 8, and 32.

In [1]:
import torch, time
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained("gpt2")
tok.pad_token = tok.eos_token
tok.padding_side = "left"

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [6]:
def bench(model, batch_size, use_cache=True, prompt="The future of physics is", n_tokens=300, runs=10):
    inputs = tok([prompt] * batch_size, return_tensors="pt", padding=True).to("cuda")
    kw = dict(do_sample=False, pad_token_id=tok.eos_token_id, use_cache=use_cache)
    full = dict(max_new_tokens=n_tokens, min_new_tokens=n_tokens, **kw)

    for i in range(10):  #warm-up
        model.generate(**inputs, **full)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    t = time.time()  #ttft
    for i in range(runs):
        model.generate(**inputs, max_new_tokens=1, **kw)
    torch.cuda.synchronize()
    ttft_ms = (time.time() - t) / runs * 1000

    t = time.time()  #tokens/second
    for i in range(runs):
        model.generate(**inputs, **full)
    torch.cuda.synchronize()
    tps = batch_size * n_tokens * runs / (time.time() - t)

    return ttft_ms, tps, torch.cuda.max_memory_allocated() / 1e6  #VRAM

In [3]:
configs = [
    ("fp32, cache on",  torch.float32, True),
    ("fp32, cache off", torch.float32, False),
    ("fp16, cache on",  torch.float16, True),
    ("fp16, cache off", torch.float16, False),
]

print("| config | batch | TTFT (ms) | tokens/sec | peak mem (MB) |")
print("|---|---|---|---|---|")
for name, dtype, cache in configs:
    model = AutoModelForCausalLM.from_pretrained("gpt2", dtype=dtype).to("cuda")
    for bs in [1, 8, 32]:
        ttft, tps, mem = bench(model, bs, use_cache=cache)
        print(f"| {name} | {bs} | {ttft:.1f} | {tps:.0f} | {mem:.0f} |")
    del model
    torch.cuda.empty_cache()

| config | batch | TTFT (ms) | tokens/sec | peak mem (MB) |
|---|---|---|---|---|


model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

| fp32, cache on | 1 | 12.3 | 97 | 533 |


KeyboardInterrupt: 